# LAB6: Prompt Engineering - Transformar

## Configuración de la API de OpenAI
Este código importa las librerías necesarias para trabajar con la API de OpenAI, carga las variables de entorno desde un archivo .env y configura la clave de la API de OpenAI para ser utilizada en las solicitudes a la API.


In [1]:
%pip install openai

In [2]:
from openai import OpenAI
import getpass

api_key = getpass.getpass("Enter your OpenAI API Key:")

client = OpenAI(api_key = api_key)

Enter your OpenAI API Key:··········


## Función para obtener respuestas de GPT
Este código define una función `get_completion` que toma un prompt y opcionalmente un modelo (por defecto `gpt-3.5-turbo`) y utiliza la API de OpenAI para obtener una respuesta. La función configura la solicitud con una temperatura de 0 para respuestas más deterministas y retorna el contenido de la respuesta.


In [3]:
def get_completion(prompt, model="gpt-4.1-nano", temperature = 0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## Traducción

In [4]:
prompt = f"""
Translate the following English text to catalan: \
```I found it is the small everyday deeds of ordinary folk that keep the darkness at bay… small acts of kindness and love.```
"""
response = get_completion(prompt)
print(response)

Vaig descobrir que són les petites accions quotidianes de la gent comú les que mantenen la foscor a ratlla... petits gestos de bondat i amor.


In [5]:
prompt = f"""
Tell me which language this is:
```Niligundua kwamba matendo madogo ya kila siku ya watu wa kawaida ndiyo yanaweka giza pembeni.```
"""
response = get_completion(prompt)
print(response)

This is Swahili.


In [6]:
prompt = f"""
Translate the following text to English pirate
```I found it is the small everyday deeds of ordinary folk that keep the darkness at bay… small acts of kindness and love.```
"""
response = get_completion(prompt)
print(response)

I be findin' it be the small everyday deeds of ordinary scallywags that keep the darkness at bay... small acts of kindness and love. Arrr!


In [21]:
prompt = f"""
Translate the following text to SQL
```Listado de usuarios que han comprado un producto por encima de 100€```
"""
response = get_completion(prompt, temperature = 0.6)
print(response)

SELECT * 
FROM usuarios
WHERE producto_precio > 100;


### ¿Cómo podemos mejorar este último prompt de Texto a SQL?


In [29]:
prompt = f"""
Translate the following text to SQL
```Listado de usuarios que han comprado un producto por encima de 100€```

Los campos de la tabla 'Users' son; id, nombre, email
Los campos de la tabla 'Sales' son; id, user_id, product_id, price, total
"""
response = get_completion(prompt, temperature = 0.8)
print(response)

SELECT Users.id, Users.nombre, Users.email
FROM Users
JOIN Sales ON Users.id = Sales.user_id
WHERE Sales.price > 100;


## Traductor universal

In [30]:
user_messages = [
    "¿Cómo puedo devolver un producto que no cumple con mis expectativas?",
    "How do I track my order that hasn't arrived yet?",
    "Le produit reçu ne correspond pas à la description sur le site.",
    "Wie kann ich ein beschädigtes Paket melden?",
    "订单延迟了怎么办？"
]

In [31]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to Catalan and Spanish:
    ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (This is Spanish.): ¿Cómo puedo devolver un producto que no cumple con mis expectativas?
Catalan: Com puc retornar un producte que no compleix les meves expectatives?
Spanish: ¿Cómo puedo devolver un producto que no cumple con mis expectativas? 

Original message (English): How do I track my order that hasn't arrived yet?
Catalan: Com puc fer seguiment del meu comanda que encara no ha arribat?
Spanish: ¿Cómo puedo hacer seguimiento de mi pedido que aún no ha llegado? 

Original message (French): Le produit reçu ne correspond pas à la description sur le site.
El producte rebut no correspon a la descripció al lloc web.
El producto recibido no corresponde a la descripción en el sitio web. 

Original message (German): Wie kann ich ein beschädigtes Paket melden?
Catalan: Com puc informar d'un paquet danyat?
Spanish: ¿Cómo puedo reportar un paquete dañado? 

Original message (This is Chinese (Simplified).): 订单延迟了怎么办？
Catalan: Com gestionar un retard en el comanda?
Spanish: ¿

## Transformación de la tonalidad


In [33]:
email = """
¡Hey colega! Gracias por el ascenso que me has dado, qué palo tener ahora que mandar
a esta pandilla de inútiles que tenemos por compañeros
"""


In [34]:
prompt = f"""
Traduce el texto entre *** a un texto muy formal y sin faltas de respeto:
***{email}***
"""
response = get_completion(prompt)
print(response)


Estimado colega, deseo expresar mi agradecimiento por la oportunidad de ascenso que me ha brindado. Agradezco la confianza depositada en mí para liderar al equipo de colaboradores que conforman nuestra organización.


## Transformación de formato

In [38]:
data_json = {
  "fish_market_employees": [
    {"name": "Joan Pere", "email": "joan.pere@peixateria.com"},
    {"name": "Rosa Mari", "email": "rosa.mari@peixateria.com"},
    {"name": "Lluís Torrent", "email": "lluis.torrent@peixateria.com"}
  ]
}

prompt = f"""
Translate the following Brainfuck code {data_json}
"""
response = get_completion(prompt)
print(response)

> +[----->+++<]>.---.+++++++..+++.[->++++<]>+.-----------.++++++++++++.-----------.-[--->+<]>-.+[->+++<]>++.++++++++++++.-----------.+++++.-------.+++++++++++++.-----------.+++++.-------.+++++++++++++.-----------.+++++.-------.


## Corrección ortográfica


In [39]:
mensajes = [
    "Fui al mediko porque me dolia la cabesa.",
    "Me gustaria tener un ordenador nuevo pa mi cumple.",
    "Emos comprado pescao pa cenar, pero estaba un poco pasao.",
    "El libro que estoy leyendo es muy interesante.",
    "Perdi las llaves de casa, no se donde las deje."
]


In [40]:
for t in mensajes:
    prompt = f"""Revisa y corrige el siguiente texto en castellano y
    reescribe la versión corregida. Si no encuentras ningún error,
    simplemente di null.
    ```{t}```"""
    response = get_completion(prompt, model="gpt-4o")
    print(response)


Fui al médico porque me dolía la cabeza.
Me gustaría tener un ordenador nuevo para mi cumpleaños.
Hemos comprado pescado para cenar, pero estaba un poco pasado.
null
Perdí las llaves de casa, no sé dónde las dejé.
